# 03 — Why Lat/Lon Is Weird

Latitude and longitude look like a coordinate grid. They plot like one. You can index and subtract them like one. For small, local work, they behave like one — just barely well enough to lull you into complacency.

Then you try to compute a distance, or a bearing, or compare features across a large area, and the numbers are wrong. Not dramatically wrong. Just consistently, quietly wrong in ways that take a while to notice.

This notebook explains why.

## 1. Why Lat/Lon Feels Like (x, y) but Isn't

The temptation is understandable:

```text
longitude increases left → right   (like x)
latitude  increases bottom → top   (like y)
```

Plot two points on a map. They appear at the right spots. Subtract their coordinates. You get a number. Everything seems fine.

The problem is what that number means. In a true Cartesian coordinate system, `(x₂ - x₁)` gives you a distance in the same units the axes are measured in. Subtract two pixel positions, you get pixels. Subtract two meter positions, you get meters.

Subtract two longitudes, you get **degrees**. Degrees are an angular unit — they describe a fraction of a rotation around the Earth's center. They are not a length. The physical distance represented by one degree changes depending on where you are on the planet.

## 2. Degrees Are Not Meters

One degree of latitude is always approximately the same physical distance — roughly **111 km** — because lines of latitude are parallel and evenly spaced.

One degree of longitude is a completely different story. Longitude lines (meridians) converge at the poles. At the equator they are farthest apart — also about 111 km per degree. But as you move toward the poles, they get closer and closer together until they meet at a single point at 90°N or 90°S, where one degree of longitude covers **zero distance**.

The formula for the ground distance per degree of longitude at a given latitude is:

```text
km_per_degree_lon = 111.32 × cos(latitude_in_radians)
```

At the equator (`lat = 0`), `cos(0) = 1.0`, so about 111 km per degree.  
At Texas (`lat ≈ 33°`), `cos(33°) ≈ 0.84`, so about 93 km per degree.  
At Greenland (`lat ≈ 72°`), `cos(72°) ≈ 0.31`, so about 34 km per degree.

In [ ]:
import math

KM_PER_DEGREE_LAT = 111.32   # roughly constant

def km_per_degree_lon(lat_degrees):
    """Approximate ground distance per degree of longitude at a given latitude."""
    return KM_PER_DEGREE_LAT * math.cos(math.radians(lat_degrees))


locations = [
    ("Equator",   0),
    ("Texas",    33),
    ("New York", 41),
    ("London",   51),
    ("Oslo",     60),
    ("Greenland",72),
    ("North Pole",90),
]

print(f"{'Location':<14} {'Latitude':>10}   {'km / 1° lon':>12}   {'km / 1° lat':>12}")
print("-" * 56)
for name, lat in locations:
    lon_km = km_per_degree_lon(lat)
    print(f"{name:<14} {lat:>10}°   {lon_km:>11.1f} km   {KM_PER_DEGREE_LAT:>11.1f} km")

## 3. Longitude Spacing Changes with Latitude

The table above shows the numbers. Now look at what this means for your data.

Take two points that are **one degree of longitude apart**:

- `(-98.0, 33.0)` and `(-97.0, 33.0)` — one degree apart in Texas
- `(-98.0, 72.0)` and `(-97.0, 72.0)` — one degree apart in Greenland

The coordinate difference is identical. But the physical distances are completely different.

In [ ]:
pairs = [
    ("Texas",     (-98.0, 33.0), (-97.0, 33.0)),
    ("New York",  (-74.0, 41.0), (-73.0, 41.0)),
    ("Oslo",      (-10.0, 60.0), ( -9.0, 60.0)),
    ("Greenland", (-98.0, 72.0), (-97.0, 72.0)),
]

print(f"{'Region':<12}  {'Δ lon':>6}  {'Approx distance':>18}")
print("-" * 44)
for region, p1, p2 in pairs:
    delta_lon = abs(p2[0] - p1[0])
    lat = p1[1]
    approx_km = delta_lon * km_per_degree_lon(lat)
    print(f"{region:<12}  {delta_lon:>5.1f}°  {approx_km:>16.1f} km")

The same angular gap — one degree — corresponds to roughly **93 km** in Texas and roughly **34 km** in Greenland. If you treat degree differences as distances, your sense of scale is off by nearly 3× between those two locations.

## 4. Flat-Earth Shortcut vs. Spherical Thinking

Given all of the above, is it ever acceptable to treat lat/lon as a flat grid?

Yes — within a narrow set of conditions.

**When the approximation is acceptable:**
- Small geographic area (a city, a county, a base)
- You only need rough spatial relationships, not precise measurements
- You are doing visualization, not navigation
- You are computing bounding boxes for display or filtering purposes

**When it is not acceptable:**
- Long-distance measurement (city to city, continental scale)
- Navigation or routing where accuracy matters
- Computing bearings or headings
- Anything where someone acts on the result

The cutoff is fuzzy, but a useful rule of thumb: **within about 50 km, naive degree-based geometry introduces less than ~1% error**. Beyond that, you should use proper geodesic formulas.

The key word is *knowing which situation you are in*. Most errors come not from using the wrong formula, but from using the flat-Earth formula without realizing it is the flat-Earth formula.

## 5. Why Bounding Boxes Still Work

After all that, you might wonder whether the bounding boxes from the previous notebooks are worthless. They are not.

A bbox is used for **containment and extent**, not distance. The questions it answers are:

- Does this point fall within these coordinate boundaries?
- What is the rough geographic footprint of this dataset?
- Which features are roughly in the same area?

None of these require converting degrees to meters. Comparing a latitude value to a latitude boundary, or a longitude to a longitude boundary, is valid — you are comparing values on the same axis, measured in the same unit.

Where a bbox breaks down is if you use it to infer distance:

```python
# This is NOT a distance — it is an angular difference
delta_lon = max_lon - min_lon
```

That difference is not "how wide" in km. It is "how many degrees wide" — and how many km that represents depends entirely on the latitude.

Use bbox for what it is good at. Do not ask it to be a ruler.

## 6. Naive Distance vs. Actual Distance

Consider two points in Texas, 1 degree of longitude apart. A naive Euclidean calculation on the raw coordinates gives one number. The actual ground distance, accounting for curvature, gives a different one.

In [ ]:
def euclidean_degrees(p1, p2):
    """Straight-line distance in degrees — not a real distance."""
    return math.sqrt((p2[0] - p1[0])**2 + (p2[1] - p1[1])**2)


def haversine_km(p1, p2):
    """
    Great-circle distance between two [lon, lat] points in kilometers.
    Uses the Haversine formula — the right tool for geographic distance.
    """
    R = 6371.0  # Earth radius in km
    lon1, lat1 = math.radians(p1[0]), math.radians(p1[1])
    lon2, lat2 = math.radians(p2[0]), math.radians(p2[1])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    return R * 2 * math.asin(math.sqrt(a))


test_cases = [
    ("Same latitude, 1° lon apart — Texas",   [-98.0, 33.0], [-97.0, 33.0]),
    ("Same latitude, 1° lon apart — Greenland", [-98.0, 72.0], [-97.0, 72.0]),
    ("Same longitude, 1° lat apart",           [-98.0, 33.0], [-98.0, 34.0]),
    ("Dallas to Houston (~360 km)",            [-96.80, 32.78], [-95.37, 29.76]),
    ("Dallas to London (~8000 km)",            [-96.80, 32.78], [ -0.13, 51.51]),
]

print(f"{'Case':<40}  {'Euclidean (°)':>14}  {'Haversine (km)':>15}")
print("-" * 74)
for label, p1, p2 in test_cases:
    naive = euclidean_degrees(p1, p2)
    real  = haversine_km(p1, p2)
    print(f"{label:<40}  {naive:>13.4f}°  {real:>14.1f} km")

Notice what Euclidean distance produces — a number in degrees with no consistent meaning. The same `1.0°` Euclidean result corresponds to ~93 km in Texas and ~34 km in Greenland. For Dallas to London, the Euclidean value is meaningless as a distance estimate entirely.

The Haversine formula is what the next module is built around. You will implement it properly there.

## 7. When Is Pretending the Earth Is Flat Acceptable?

This is a mature engineering question and the answer is: *it depends on your required accuracy and your geographic scope.*

The table below shows approximate Euclidean error as a percentage of the true Haversine distance, for points at ~33° latitude (Texas):

In [ ]:
import math

# Base point in Texas
base = [-98.0, 33.0]

# Generate a point at increasing distances by stepping east (longitude only)
# so the comparison is clean
offsets_deg = [0.01, 0.05, 0.1, 0.5, 1.0, 3.0, 5.0, 10.0]

print(f"{'Offset':>8}  {'Haversine':>12}  {'Euclidean·cos':>15}  {'Error':>8}")
print("-" * 52)

for d in offsets_deg:
    target = [base[0] + d, base[1]]
    hav = haversine_km(base, target)

    # A slightly better naive approximation: scale longitude by cos(lat)
    dlat = target[1] - base[1]
    dlon = (target[0] - base[0]) * math.cos(math.radians(base[1]))
    naive_scaled = math.sqrt(dlat**2 + dlon**2) * KM_PER_DEGREE_LAT

    err = abs(naive_scaled - hav) / hav * 100
    print(f"{d:>7.2f}°  {hav:>11.2f} km  {naive_scaled:>14.2f} km  {err:>7.2f}%")

Even the scaled Euclidean approximation (which corrects for longitude compression at a given latitude) accumulates error as distances grow. At small scales the error is negligible. At 10° (roughly 930 km) it becomes significant.

The Haversine formula has no such degradation — it is exact on a sphere regardless of scale.

---

## Exercise A — Explain the Gap

The two points `(-98.0, 33.0)` and `(-97.0, 33.0)` are exactly `1.0` apart by Euclidean degree distance.

In plain English (no formulas required), explain why they are **not** exactly one unit apart in any physically meaningful sense. What would you need to know to convert that `1.0` into kilometers?

In [ ]:
# your answer here (as a comment or print statement)


## Exercise B — One Degree of Longitude Across Latitudes

Using `km_per_degree_lon`, compute the approximate ground distance covered by one degree of longitude at every 10° of latitude from 0° to 80°. Print the results as a table.

In [ ]:
# your code here
# hint: range(0, 81, 10)

## Exercise C — Acceptable vs. Not Acceptable

For each task below, decide whether using raw lat/lon degree arithmetic is acceptable or whether a proper geodesic formula is required. Write a one-sentence justification for each.

| Task | Acceptable? | Why? |
|------|-------------|------|
| Check whether a point falls inside a bbox | ? | |
| Compute the distance between two cities 500 km apart | ? | |
| Determine which of ten features is "nearest" to a base, all within a 10 km radius | ? | |
| Compute the bbox of a FeatureCollection for display purposes | ? | |
| Calculate a bearing for a flight path from Dallas to London | ? | |
| Filter a 100,000-feature dataset to a rough region of interest | ? | |

## Exercise D — Reflection

Write 3–5 sentences answering this question:

> A missile guidance system uses latitude and longitude to track a target. Why does it matter — potentially a great deal — that Earth is not flat?

Think about: what happens to the error as distance increases, what bearing means on a curved surface, and what the consequence of even a small percentage error is at operational ranges.

In [ ]:
# your answer here

---

## Check Your Understanding

A classmate writes this function to find the nearest point in a list to a given location:

```python
def nearest(target, points):
    return min(points, key=lambda p: euclidean_degrees(target, p))
```

They test it on a dataset of locations spread across the continental United States and the results seem reasonable. Then they run the same function on a dataset spanning from Texas to Scandinavia and the results are quietly wrong.

**Question:** Why does this function work acceptably for a US-only dataset but produce incorrect results when the dataset spans a much larger area? What specifically goes wrong, and how would you fix it?

```python
# your answer here
```


---

## Next

In [06 — Distance](../06-Distance/00-Euclidean_Distance.ipynb), we implement the Haversine formula from scratch — the tool that solves everything this notebook identified as broken about degree arithmetic.